# Real Estate EDA

Basic exploratory data analysis for the model-ready real estate CSV.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

## Load Data

Use the compact model-feature CSV for EDA.

In [ ]:
DATA_PATH = Path("..") / "data" / "scraped_real_estate_model_features.csv"

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df.info()

## Missing Values

Check which columns need cleaning before modeling.

In [ ]:
missing = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_percent=lambda x: (x["missing_count"] / len(df) * 100).round(2))
    .sort_values("missing_count", ascending=False)
)

missing.head(20)

## Target Price

Review the price target and outliers.

In [ ]:
df["price_usd"].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df["price_usd"], bins=40, ax=axes[0])
axes[0].set_title("Price Distribution")

sns.histplot(df[df["price_usd"] <= df["price_usd"].quantile(0.95)]["price_usd"], bins=40, ax=axes[1])
axes[1].set_title("Price Distribution Under 95th Percentile")

plt.tight_layout()

## Property Mix

Compare apartments and houses.

In [ ]:
df["property_type"].value_counts(dropna=False)

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="property_type", order=df["property_type"].value_counts().index)
plt.title("Rows by Property Type")
plt.xlabel("Property Type")
plt.ylabel("Rows")
plt.tight_layout()

In [ ]:
df.groupby("property_type")["price_usd"].describe().round(2)

## Numeric Features

Look at basic distributions and correlations.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
df[numeric_cols].describe().T.round(2)

In [ ]:
corr = df[numeric_cols].corr(numeric_only=True)["price_usd"].sort_values(ascending=False)
corr.head(15)

In [ ]:
top_corr = corr.abs().sort_values(ascending=False).head(12).index

plt.figure(figsize=(10, 7))
sns.heatmap(df[top_corr].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Top Numeric Correlations")
plt.tight_layout()

## Simple Feature Checks

Check important residence features against price.

In [ ]:
important_features = [
    "effective_area_sqm",
    "bedrooms",
    "bathrooms",
    "parking_spaces",
    "amenity_count",
]

fig, axes = plt.subplots(1, len(important_features), figsize=(20, 4))

for ax, col in zip(axes, important_features):
    sns.scatterplot(data=df, x=col, y="price_usd", hue="property_type", alpha=0.5, ax=ax)
    ax.set_title(col)
    ax.legend_.remove() if ax.legend_ else None

plt.tight_layout()

## Notes

- Handle missing values before modeling.
- Review high price outliers.
- Encode categorical columns before training.